In [ ]:
with open('../data/archive/1of2/wiki_00', 'r', encoding='utf-8') as f:
    text = f.read()

print("The length of our dataset: ", len(text))


The length of dataset:  1030267


In [2]:
# Here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"$%&'()*+,-./0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[]^_`abcdefghijklmnopqrstuvwxyz|~ £°²³´·¹ÁÅÖ×àáâãäåæçèéêìíîïðñòóôöøúüÿāēĕěīıłōūǫəɪʻʿˈΓΔΜάέίαβγδεηθικλμνορςστυφωόύابجرشلهἀἄἡἥἶ‎–—‘’“”†…€ℓ→−♆の万八大姬燕百神蓟都
207


In [4]:
# Tokenizer! (Extremely Simple Char-level Tokenizer)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [ stoi[c] for c in s] # string to integers
decode = lambda l: ''.join(itos[i] for i in l) # integers to string

print(encode("I need to eat something"))
print(decode(encode("I need to eat something")))

[40, 1, 76, 67, 67, 66, 1, 82, 77, 1, 67, 63, 82, 1, 81, 77, 75, 67, 82, 70, 71, 76, 69]
I need to eat something


In [5]:
# Let's now encode the entire text dataset and store it into a torch.Tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1030267]) torch.int64
tensor([32, 78, 80, 71, 74,  0,  0, 32, 78, 80, 71, 74,  1,  8, 32, 78, 80, 14,
         9,  1, 71, 81,  1, 82, 70, 67,  1, 68, 77, 83, 80, 82, 70,  1, 75, 77,
        76, 82, 70,  1, 77, 68,  1, 82, 70, 67,  1, 87, 67, 63, 80,  1, 71, 76,
         1, 82, 70, 67,  1, 41, 83, 74, 71, 63, 76,  1, 63, 76, 66,  1, 38, 80,
        67, 69, 77, 80, 71, 63, 76,  1, 65, 63, 74, 67, 76, 66, 63, 80, 81, 12,
         1, 63, 76, 66,  1, 65, 77, 75, 67, 81,  1, 64, 67, 82, 85, 67, 67, 76,
         1, 44, 63, 80, 65, 70,  1, 63, 76, 66,  1, 44, 63, 87, 14,  1, 40, 82,
         1, 71, 81,  1, 77, 76, 67,  1, 77, 68,  1, 68, 77, 83, 80,  1, 75, 77,
        76, 82, 70, 81,  1, 82, 77,  1, 70, 63, 84, 67,  1, 19, 16,  1, 66, 63,
        87, 81, 14,  0,  0, 32, 78, 80, 71, 74,  1, 63, 74, 85, 63, 87, 81,  1,
        64, 67, 69, 71, 76, 81,  1, 77, 76,  1, 82, 70, 67,  1, 81, 63, 75, 67,
         1, 66, 63, 87,  1, 77, 68,  1, 82, 70, 67,  1, 85, 67, 67, 73,  1, 63,
      

In [ ]:
# Let's now split up the data into train and validation set
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
# We need to sample a random chunk of data for the sake of efficient (possible) training 
# The following is the size of the chunk
block_size = 4
train_data[:block_size+1]

tensor([32, 78, 80, 71, 74])

In [9]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When input is {context}, the target: {target}")

When input is tensor([32]), the target: 78
When input is tensor([32, 78]), the target: 80
When input is tensor([32, 78, 80]), the target: 71
When input is tensor([32, 78, 80, 71]), the target: 74


In [ ]:
# We need to consider batch size then (a set of independent chuncks to exploit the parallelism of GPU)
torch.manual_seed(4234)
batch_size = 4 # how many independent sequences (c.f. "chunks") will we process in parallel?
block_size = 8 # what's the maximum context length for predictions? 

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # randonly pick a sample from the training set
    x = torch.stack([data[i:i+block_size] for i in ix]) # torch.stack: stacking 1-dim tensor to 2-dim tensor (literally 'stacking' them)  
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print("---")

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 1, 65, 83, 80, 84, 67,  1, 63],
        [71, 81,  1, 13, 17, 14,  0,  0],
        [74,  1, 65, 77, 80, 80, 67, 65],
        [76, 63, 12,  1, 63, 76, 66,  1]])
targets:
torch.Size([4, 8])
tensor([[65, 83, 80, 84, 67,  1, 63, 82],
        [81,  1, 13, 17, 14,  0,  0, 51],
        [ 1, 65, 77, 80, 80, 67, 65, 82],
        [63, 12,  1, 63, 76, 66,  1, 82]])
---
when input is [1] the target: 65
when input is [1, 65] the target: 83
when input is [1, 65, 83] the target: 80
when input is [1, 65, 83, 80] the target: 84
when input is [1, 65, 83, 80, 84] the target: 67
when input is [1, 65, 83, 80, 84, 67] the target: 1
when input is [1, 65, 83, 80, 84, 67, 1] the target: 63
when input is [1, 65, 83, 80, 84, 67, 1, 63] the target: 82
when input is [71] the target: 81
when input is [71, 81] the target: 1
when input is [71, 81, 1] the target: 13
when input is [71, 81, 1, 13] the target: 17
when input is [71, 81, 1, 13, 17] the target: 14
when input is [71, 81, 1

In [ ]:
# Now we have a input batch for our transformer
print(xb)

tensor([[ 1, 65, 83, 80, 84, 67,  1, 63],
        [71, 81,  1, 13, 17, 14,  0,  0],
        [74,  1, 65, 77, 80, 80, 67, 65],
        [76, 63, 12,  1, 63, 76, 66,  1]])


In [ ]:
# Let's look through Bigram Language Model first
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337) # 1337 is just a convention or for the sake of reproductability on any other envs.

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        
        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx) # (Batch, Time, Channel-vocab_size)

        # cross_entropy() wants the format of follows 
        if targets is None:
            loss = None
        else :     
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    # idx: context 
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last step
            logits = logits[:, -1,:] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sample index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
        


m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
# What went wrong?
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

# Currently the history isn't used at all.
# It generates totally random values, we need to train it.

torch.Size([32, 207])
tensor(5.8032, grad_fn=<NllLossBackward0>)

37DWشÁςΓ%':ν×ø♆>φωã
ε‘ī¹´هł£z(αjι<ü燕رφàfνô8ÿέə7β百[ìÿ,´°´*蓟万WðāU.αäشℓ5]ěἄόἡφθIιل–αbج–八rUのī‘八ʿ0ιPˈρ0QΔ


In [24]:
# Create a PyTorch optimizer
optimizer = torch.optim.Adam(m.parameters(), lr=1e-3)

In [29]:
# Configure the batch
batch_size = 32
for step in range(10000):
    # Sample a batch of data (for training)
    xb, yb = get_batch('train')

    # Evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    print(loss.item())

2.4436910152435303
2.529599905014038
2.441232919692993
2.466231346130371
2.4714837074279785
2.407040596008301
2.4269392490386963
2.5415897369384766
2.3073830604553223
2.4779598712921143
2.400129556655884
2.583153009414673
2.5732061862945557
2.4855496883392334
2.5136702060699463
2.5213420391082764
2.537527084350586
2.3807735443115234
2.3042168617248535
2.3481671810150146
2.5796408653259277
2.4061577320098877
2.559654951095581
2.4924557209014893
2.657782793045044
2.436234951019287
2.268690347671509
2.520347833633423
2.377772092819214
2.5095715522766113
2.3699660301208496
2.475257396697998
2.444211721420288
2.4768497943878174
2.4924025535583496
2.6001360416412354
2.528141498565674
2.464193105697632
2.4816055297851562
2.4250547885894775
2.4766693115234375
2.4793875217437744
2.5065155029296875
2.360574245452881
2.452781915664673
2.521071434020996
2.526878595352173
2.3577380180358887
2.4018349647521973
2.4786672592163086
2.7328972816467285
2.558534622192383
2.5290417671203613
2.5952160358428

In [ ]:
# After the minimal training, we can get this:
print(decode(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=400)[0].tolist()))


Inned n iofutisp or d Sphome cy ny iooveve a An wo cand, Seerandethegopas secarlarlatwoumatin obee the tilolleped The ofrois) h mendolstoul telintcuss. qung evion chenmay's, s tarigso tharony at trisuatha ime tofoudolles. s by halod cudy ben e andatlod ithe tothombearl Ledichhecish 1601. y beal Thee Herurex bed ousod fotofithe a g tes. eocecis bareas (3 tithepoumetogres ofaed inf &Juizangn urefes 


# Suggested exercises:

EX1: The n-dimensional tensor mastery challenge: Combine the `Head` and `MultiHeadAttention` into one class that processes all the heads in parallel, treating the heads as another batch dimension (answer is in nanoGPT).

EX2: Train the GPT on your own dataset of choice! What other data could be fun to blabber on about? (A fun advanced suggestion if you like: train a GPT to do addition of two numbers, i.e. a+b=c. You may find it helpful to predict the digits of c in reverse order, as the typical addition algorithm (that you're hoping it learns) would proceed right to left too. You may want to modify the data loader to simply serve random problems and skip the generation of train.bin, val.bin. You may want to mask out the loss at the input positions of a+b that just specify the problem using y=-1 in the targets (see CrossEntropyLoss ignore_index). Does your Transformer learn to add? Once you have this, swole doge project: build a calculator clone in GPT, for all of +-*/. Not an easy problem. You may need Chain of Thought traces.)

EX3: Find a dataset that is very large, so large that you can't see a gap between train and val loss. Pretrain the transformer on this data, then initialize with that model and finetune it on tiny shakespeare with a smaller number of steps and lower learning rate. Can you obtain a lower validation loss by the use of pretraining?

EX4: Read some transformer papers and implement one additional feature or change that people seem to use. Does it improve the performance of your GPT?